# Notebook 01 — Core LLM Workflow

Demonstrates provider swap, non-determinism, cache observability, prompting basics, and clean failure surfacing against a real Anthropic backend.

<!-- TODO main-session: expand teaching framing -->


## Setup

Imports and environment loading.


In [3]:
from __future__ import annotations
import os
import sys
from pathlib import Path

# Add llmops-session to path so `from src.llm import ...` works from notebooks/
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    load_dotenv = None

if load_dotenv is not None:
    load_dotenv(repo_root / ".env")

from src.llm import LLMClient, LLMCache, CompletionResult

# Confirm we have an Anthropic key (warn but don't fail if mock)
provider = os.getenv("LLM_PROVIDER", "anthropic")
has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Provider: {provider} · Anthropic key present: {has_key}")


Provider: anthropic · Anthropic key present: True


## Provider abstraction

Same `.complete(...)` call shape, different provider implementations.


In [4]:
prompt = "In one sentence, what is LLM Ops?"

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — set ANTHROPIC_API_KEY to run]")
else:
    client = LLMClient()
    result = client.complete(prompt, max_tokens=80)
    print("provider:", result.provider)
    print("model:", result.model)
    print("text:", result.text)


provider: anthropic
model: claude-haiku-4-5-20251001
text: LLM Ops refers to the operational practices and tools for deploying, monitoring, maintaining, and optimizing large language models in production environments.


In [5]:
prompt = "In one sentence, what is LLM Ops?"

client = LLMClient(provider="mock")
result = client.complete(prompt, max_tokens=80)
print("provider:", result.provider)
print("model:", result.model)
print("text:", result.text)


provider: mock
model: mock-model-v1
text: [mock:b8beac3c] echo: In one sentence, what is LLM Ops?


In [6]:
prompt = "In one sentence, what is LLM Ops?"

for provider_name, env_var in [("openai", "OPENAI_API_KEY"), ("gemini", "GOOGLE_API_KEY"), ("ollama", None)]:
    if env_var is not None and not os.getenv(env_var):
        print(f"{provider_name}: [skipped — missing {env_var}]")
        continue

    try:
        client = LLMClient(provider=provider_name)
        result = client.complete(prompt, max_tokens=80)
        preview = " ".join(result.text.split())[:80]
        print(f"{provider_name}: {preview}")
    except ImportError as exc:
        print(f"{provider_name}: [skipped — {exc}]")
    except Exception as exc:
        print(f"{provider_name}: [skipped — {type(exc).__name__}: {exc}]")


openai: [skipped — missing OPENAI_API_KEY]
gemini: [skipped — missing GOOGLE_API_KEY]
ollama: [skipped — ConnectionError: Ollama server not reachable at http://localhost:11434. Is `ollama serve` running?]


The only line that changed across cells 4–6 is the provider name.


## Non-determinism and system messages

Repeated hosted calls can vary, and system messages steer response style.


In [7]:
variance_prompt = "List three risks of deploying an LLM in production. Be specific."

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no key]")
else:
    client = LLMClient(cache=False)
    for run_number in range(1, 4):
        result = client.complete(variance_prompt, cache=False, temperature=0.9, max_tokens=160)
        print(f"Run {run_number}:")
        print(result.text)
        print()


Run 1:
# Three Production LLM Risks

## 1. **Hallucination and Factual Inaccuracy**
LLMs generate plausible-sounding text that can contain false information presented with confidence. In production, this creates liability—a financial advisory chatbot might give incorrect investment guidance, or a medical chatbot could suggest dangerous treatments. Unlike traditional software bugs with deterministic failures, hallucinations are probabilistic and difficult to catch exhaustively during testing.

## 2. **Prompt Injection and Security Vulnerabilities**
Attackers can craft inputs that override system instructions or expose sensitive data. For example, a customer service bot might leak internal knowledge base contents if a user includes malicious instructions in their query. If the LL

Run 2:
# Three Production LLM Risks

## 1. **Hallucination-Induced Misinformation**
LLMs confidently generate plausible-sounding but factually incorrect information. In production, this creates liability—a fina

In [8]:
user_prompt = "Explain the operational risk of letting an LLM call production tools."
sre_system = "You are a terse senior SRE. Reply in one sentence and use plain language."

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no key]")
else:
    client = LLMClient(cache=False)
    plain = client.complete(user_prompt, cache=False, max_tokens=160)
    steered = client.complete(user_prompt, system=sre_system, cache=False, max_tokens=80)

    print("No system message:")
    print(plain.text)
    print()
    print("With system message:")
    print(steered.text)


No system message:
# Operational Risks of LLMs Calling Production Tools

## Core Vulnerabilities

**Hallucination & Confidence Mismatch**
- LLMs can confidently generate plausible-looking but incorrect commands
- May call non-existent endpoints or pass malformed parameters
- No inherent understanding that production systems have real consequences

**Lack of State Awareness**
- No persistent memory of system state between calls
- Can't reliably track what it just did or dependencies
- May issue contradictory commands in sequence (e.g., scale up then immediately down)

## High-Impact Failure Modes

| Risk | Example | Impact |
|------|---------|--------|
| **Destructive

With system message:
An LLM calling production tools can execute unintended actions due to hallucinations, prompt injection, or misunderstanding context, causing data loss, service outages, or security breaches before humans can intervene.


## Cache hit / miss / version

Cache events expose misses, hits, and prompt-version invalidation.


In [9]:
if not os.getenv("ANTHROPIC_API_KEY"):
    cache_demo_ready = False
    print("[skipped — no key]")
else:
    cache_demo_ready = True
    events: list[str] = []
    cache = LLMCache(on_cache_event=lambda status: events.append(status))
    client = LLMClient(cache=cache, prompt_version="v1")

    prompt = "What's the capital of France?"
    r1 = client.complete(prompt, max_tokens=32)  # expect miss
    r2 = client.complete(prompt, max_tokens=32)  # expect hit
    print("After two identical calls:", events)  # ['miss', 'hit']
    print("r1.cache_status:", r1.cache_status, "· r2.cache_status:", r2.cache_status)


After two identical calls: ['miss', 'hit']
r1.cache_status: miss · r2.cache_status: hit


In [10]:
if not globals().get("cache_demo_ready", False):
    print("[skipped — no key]")
else:
    client_v2 = LLMClient(cache=cache, prompt_version="v2")
    r3 = client_v2.complete(prompt, max_tokens=32)  # expect miss (key changed)
    print("After bumping prompt_version:", events)  # ['miss', 'hit', 'miss']
    print("r3.cache_status:", r3.cache_status)


After bumping prompt_version: ['miss', 'hit', 'miss']
r3.cache_status: miss


## Failure surfaces cleanly

Deliberately invalid credentials should raise a visible error without aborting the notebook.


In [ ]:
import os

saved = os.environ.get("ANTHROPIC_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = "invalid_key_for_demo_purposes"

try:
    bad_client = LLMClient(provider="anthropic")
    result = bad_client.complete("This call will fail.", max_tokens=16)
    print("Unexpected success:", result.text)
except Exception as e:
    print(f"Caught {type(e).__name__}: {e}")
finally:
    if saved is not None:
        os.environ["ANTHROPIC_API_KEY"] = saved
    else:
        os.environ.pop("ANTHROPIC_API_KEY", None)


## Few-shot examples

Examples tighten the output format for the same participant-message classification task.


In [ ]:
message_to_classify = "When is the deadline for assignment 2?"
labels = "policy_question, schedule_question, assignment_question, out_of_scope"

prompt_without_examples = (
    f"Classify this participant message into one of: {labels}. Reply with one label. "
    f"Message: '{message_to_classify}'"
)

prompt_with_examples = f"""Classify each participant message into one of: {labels}.
Reply with one label only.

Message: 'When does the Thursday live session start?'
Label: schedule_question

Message: 'Can you help me write my company's leave policy?'
Label: out_of_scope

Message: '{message_to_classify}'
Label:"""

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no key]")
else:
    client = LLMClient(cache=False)
    no_examples = client.complete(prompt_without_examples, cache=False, max_tokens=24)
    with_examples = client.complete(prompt_with_examples, cache=False, max_tokens=24)

    print("No examples:")
    print(no_examples.text)
    print()
    print("With examples:")
    print(with_examples.text)


## Structured output (JSON)

The same classification task becomes easier to parse when the prompt asks for a schema.


In [ ]:
import json

message_to_classify = "When is the deadline for assignment 2?"
json_prompt = f"""You are the TalentSprint program assistant triage helper.
Classify the participant message into exactly one of these labels:
policy_question, schedule_question, assignment_question, out_of_scope.

Participant message: '{message_to_classify}'

Respond as strict JSON with this schema:
{{"category": "<one of the four>", "confidence": 0.0, "reason": "<one short sentence>"}}
Do not wrap the JSON in markdown fences.
"""

if not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no key]")
else:
    client = LLMClient(cache=False)
    result = client.complete(json_prompt, cache=False, max_tokens=120)
    candidate = result.text.strip()
    if candidate.startswith("```"):
        candidate = "\n".join(candidate.splitlines()[1:-1]).strip()
    try:
        parsed = json.loads(candidate)
        print(parsed)
    except json.JSONDecodeError as exc:
        print(f"JSON parse failed: {exc}")
        print(result.text)


## Context grounding

Pasting the policy document constrains the answer to the program's actual rule.


In [ ]:
data_dir = Path(repo_root) / "data"
policy_path = data_dir / "sample_program_policy.md"

if policy_path.exists():
    chosen_path = policy_path
else:
    candidates = sorted(data_dir.glob("sample_*.md")) if data_dir.exists() else []
    chosen_path = candidates[0] if candidates else None

if chosen_path is None:
    print("[skipped — no policy doc]")
elif not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no key]")
else:
    participant_question = "A participant asks: is there a grace period for late submissions in this program?"
    policy_text = chosen_path.read_text(encoding="utf-8")
    cold_prompt = f"{participant_question} Reply in 2 sentences."
    grounded_prompt = f"""Program policy document:
---
{policy_text}
---

{participant_question}
Answer only from the document and cite the policy section name. Reply in 2 sentences.
"""

    client = LLMClient(cache=False)
    cold = client.complete(cold_prompt, cache=False, max_tokens=120)
    grounded = client.complete(grounded_prompt, cache=False, max_tokens=180)

    print(f"Policy doc: {chosen_path.name}")
    print("Cold answer:")
    print(cold.text)
    print()
    print("Grounded answer:")
    print(grounded.text)


## Prompts are artifacts — bump prompt_version

Cells 12–13 already showed that `prompt_version` is part of the cache key.
If you revise a prompt without bumping the version, the cache can serve stale answers that hide your prompt change.


## Putting it all together

One real participant question. Everything we just learned, applied.
The final answer is grounded, structured, traced — but the model is still just answering from one document we pasted in. NB 02 (RAG) scales this to many documents the model retrieves automatically.


In [ ]:
import json

participant_question = "A participant asks: can I get extra time on assignment 2 if I'm sick?"
system_msg = (
    "You are the TalentSprint program assistant. "
    "Answer only using the program documents provided. "
    "If the documents don't contain the answer, say so explicitly. "
    "Be concise and cite which document section your answer comes from."
)

policy_path = Path(repo_root) / "data" / "sample_program_policy.md"

if not policy_path.exists():
    print("[skipped — no policy doc]")
elif not os.getenv("ANTHROPIC_API_KEY"):
    print("[skipped — no ANTHROPIC_API_KEY]")
else:
    policy_text = policy_path.read_text(encoding="utf-8")
    user_prompt = f"""Program policy document:
---
{policy_text}
---

Question: {participant_question}

Respond as JSON with this schema:
{{"answer": "<your answer, citing the policy section>",
  "grounded": true/false,
  "source_section": "<section name from the document, or null>",
  "confidence": 0.0 to 1.0}}
Do not wrap the JSON in markdown fences.
"""

    events: list[str] = []
    cache = LLMCache(on_cache_event=lambda s: events.append(s))
    client = LLMClient(cache=cache, prompt_version="program_assistant_v1")
    result = client.complete(user_prompt, system=system_msg, max_tokens=220)

    print("=" * 60)
    print(f"Question: {participant_question}")
    print("=" * 60)
    candidate = result.text.strip()
    if candidate.startswith("```"):
        candidate = "\n".join(candidate.splitlines()[1:-1]).strip()
    try:
        parsed = json.loads(candidate)
        print(f"Answer:        {parsed.get('answer')}")
        print(f"Grounded:      {parsed.get('grounded')}")
        print(f"Source:        {parsed.get('source_section')}")
        print(f"Confidence:    {parsed.get('confidence')}")
    except json.JSONDecodeError:
        print(f"(JSON parse failed; raw text:)\n{result.text}")

    print()
    print(f"Provider:      {result.provider}")
    print(f"Model:         {result.model}")
    print(f"Cache status:  {result.cache_status}")
    print(f"Latency (ms):  {result.latency_ms:.0f}")
    print(f"Tokens in/out: {result.tokens_in} / {result.tokens_out}")
    print(f"Cost (USD):    ${result.cost_estimate_usd:.6f}")
    print(f"Cache events:  {events}")


## What NB 02 will add

In cell 24 we hand-picked the one document to paste in. In a real system we will not know which document has the answer, and there may be hundreds.
NB 02 builds the retrieval layer that picks the right document(s) automatically, and shows what happens when it picks the wrong ones.
